Phase 3 - Image Summary Pipeline

Team 15 - ArrayOfSunshine

OCR reads text, VLM writes Vietnamese summary.
VLM runs as Docker container via vLLM.

In [1]:
import os
os.environ["FLAGS_use_mkldnn"] = "0"

import numpy as np
import os
# NumPy < 2.0 compat fixes for PP-OCRv6
import PIL
if not hasattr(PIL, '_util'):
    class _util: pass
    PIL._util = _util
if not hasattr(PIL._util, 'is_directory'):
    PIL._util.is_directory = os.path.isdir
if not hasattr(np, 'sctypes'):
    np.sctypes = {
        "float": [np.float16, np.float32, np.float64],
        "int": [np.int8, np.int16, np.int32, np.int64],
        "uint": [np.uint8, np.uint16, np.uint32, np.uint64],
        "complex": [np.complex64, np.complex128]
    }

## Installation

In [2]:
import os
def install_if_missing(import_name, pip_command):
    try:
        __import__(import_name)
        print(f"'{import_name}' Existed")
    except ImportError:
        print(f"Installing: {pip_command}")
        from IPython import get_ipython
        ipython = get_ipython()
        if ipython is not None:
            ipython.run_line_magic('pip', f'install {pip_command}')
        else:
            os.system(f'pip install {pip_command}')

install_if_missing("cv2", "opencv-python")
install_if_missing("paddleocr", "paddleocr>=3.7.0 paddlepaddle")
install_if_missing("vietocr", "vietocr")
install_if_missing("openai", "openai")
install_if_missing("numpy", "numpy<2.0")

'cv2' Existed
'paddleocr' Existed
'vietocr' Existed
'openai' Existed
'numpy' Existed


## Imports

In [3]:
import os, json, re, time, base64, unicodedata, traceback
from io import BytesIO
from queue import Queue
from threading import Thread, Lock

import numpy as np
import pandas as pd
import cv2
try:
    import torch
    HAS_GPU = torch.cuda.is_available()
except Exception:
    import warnings
    warnings.warn("torch failed - CPU mode")
    torch = None
    HAS_GPU = False
from PIL import Image
from paddleocr import PaddleOCR
from vietocr.tool.predictor import Predictor
from vietocr.tool.config import Cfg
from tqdm import tqdm
from openai import OpenAI

# NumPy < 2.0 compat fixes for PP-OCRv6
import PIL
if not hasattr(PIL, '_util'):
    class _util: pass
    PIL._util = _util
if not hasattr(PIL._util, 'is_directory'):
    PIL._util.is_directory = os.path.isdir
if not hasattr(np, 'sctypes'):
    np.sctypes = {
        "float": [np.float16, np.float32, np.float64],
        "int": [np.int8, np.int16, np.int32, np.int64],
        "uint": [np.uint8, np.uint16, np.uint32, np.uint64],
        "complex": [np.complex64, np.complex128]
    }
print('All libraries loaded')

OSError: [WinError 127] The specified procedure could not be found. Error loading "c:\Users\black\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\lib\shm.dll" or one of its dependencies.

## Load Dataset

In [ ]:
TEST_IMAGE_FOLDER = "C:/HCMUT/Projects/HACKATHON/2nd_URA/images"
EXTENSIONS = (".jpg", ".jpeg", ".png")
image_files = sorted([
    f for f in os.listdir(TEST_IMAGE_FOLDER)
    if f.lower().endswith(EXTENSIONS)
])
print(f"Dataset : {TEST_IMAGE_FOLDER}")
print(f"Images  : {len(image_files)} files")
if image_files:
    print(f"  Range : {image_files[0]} ~ {image_files[-1]}")
print("Inference only - no training data loaded.")

## Preprocessing (adaptive image enhancement)

In [ ]:
def classify_image(img_bgr):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    mean_brightness = gray.mean()
    overexposed_ratio = (gray > 240).sum() / gray.size
    underexposed_ratio = (gray < 30).sum() / gray.size
    contrast = gray.std()
    laplacian_var = cv2.Laplacian(gray, cv2.CV_64F).var()
    saturation_std = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)[:, :, 1].std()
    if mean_brightness > 200 or overexposed_ratio > 0.30:
        return "overexposed"
    if mean_brightness < 60 or underexposed_ratio > 0.30:
        return "underexposed"
    if laplacian_var < 100:
        return "blurry"
    if contrast < 42:
        return "low_contrast"
    if saturation_std > 75:
        return "complex"
    return "normal"

def gamma_correct(img_bgr, gamma=1.3):
    inv_gamma = 1.0 / gamma
    table = np.array([(i / 255.0) ** inv_gamma * 255 for i in range(256)]).astype("uint8")
    return cv2.LUT(img_bgr, table)

def preprocess(img_pil, max_dim=1536, min_dim=800):
    img_bgr = np.array(img_pil.convert('RGB'))[:, :, ::-1]
    h, w = img_bgr.shape[:2]
    if max(h, w) > max_dim:
        scale = max_dim / max(h, w)
    elif max(h, w) < min_dim:
        scale = min_dim / max(h, w)
    else:
        scale = 1.0
    new_h, new_w = int(round(h * scale)), int(round(w * scale))
    img_bgr = cv2.resize(img_bgr, (new_w, new_h), interpolation=cv2.INTER_CUBIC)
    category = classify_image(img_bgr)
    k = np.array([[0, -0.5, 0], [-0.5, 3, -0.5], [0, -0.5, 0]])
    if category == "normal":
        lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        lab[:,:,0] = clahe.apply(lab[:,:,0])
        img_bgr = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)
        img_bgr = cv2.filter2D(img_bgr, -1, k)
    elif category == "overexposed":
        lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
        clahe = cv2.createCLAHE(clipLimit=3.5, tileGridSize=(8, 8))
        lab[:,:,0] = clahe.apply(lab[:,:,0])
        img_bgr = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)
        img_bgr = gamma_correct(img_bgr, gamma=0.7)
    elif category == "underexposed":
        lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
        clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
        lab[:,:,0] = clahe.apply(lab[:,:,0])
        img_bgr = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)
        img_bgr = gamma_correct(img_bgr, gamma=1.5)
    elif category == "low_contrast":
        lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
        lab[:,:,0] = cv2.normalize(lab[:,:,0], None, 0, 255, cv2.NORM_MINMAX)
        clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
        lab[:,:,0] = clahe.apply(lab[:,:,0])
        img_bgr = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)
        img_bgr = cv2.filter2D(img_bgr, -1, k)
    elif category == "blurry":
        gaussian = cv2.GaussianBlur(img_bgr, (0, 0), sigmaX=3)
        img_bgr = cv2.addWeighted(img_bgr, 1.5, gaussian, -0.5, 0)
        lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        lab[:,:,0] = clahe.apply(lab[:,:,0])
        img_bgr = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)
    elif category == "complex":
        img_bgr = cv2.bilateralFilter(img_bgr, d=9, sigmaColor=75, sigmaSpace=75)
        lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
        clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
        lab[:,:,0] = clahe.apply(lab[:,:,0])
        img_bgr = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)
        img_bgr = cv2.filter2D(img_bgr, -1, k)
    return img_bgr[:, :, ::-1]

def postprocess_ocr(text):
    if not text: return ""
    text = re.sub(r"\s+", " ", text).strip()
    tokens = text.split()
    if not tokens: return ""
    deduped = [tokens[0]]
    for tok in tokens[1:]:
        if tok.lower() != deduped[-1].lower():
            deduped.append(tok)
    return " ".join(deduped)
print('Preprocessing loaded')

## OCR - PP-OCRv6 detection + VietOCR recognition

In [ ]:
Detector = PaddleOCR(
    lang='vi', det=True, rec=False,
    ocr_version='PP-OCRv4',
    use_gpu=torch.cuda.is_available(),
    show_log=False,
    det_db_box_thresh=0.3,
    det_db_unclip_ratio=2.0,
    det_limit_side_len=1536,
)
print(f'PP-OCRv6 loaded. GPU: {torch.cuda.is_available()}')

In [ ]:
config = Cfg.load_config_from_name('vgg_transformer')
config['device'] = 'cuda:0' if torch.cuda.is_available() else 'cpu'
config['predictor']['beamsearch'] = False
recognizer = Predictor(config)
print('VietOCR loaded')

In [ ]:
def Crop_Padding(image, bbox, pad=8):
    box = np.array(bbox, dtype=int)
    x_min = max(0, np.min(box[:, 0]) - pad)
    x_max = min(image.shape[1], np.max(box[:, 0]) + pad)
    y_min = max(0, np.min(box[:, 1]) - pad)
    y_max = min(image.shape[0], np.max(box[:, 1]) + pad)
    return image[y_min:y_max, x_min:x_max]

def Sort_Boxes(boxes):
    if not boxes: return []
    boxes = sorted(boxes, key=lambda b: b[0][1])
    heights = [abs(b[2][1] - b[0][1]) for b in boxes]
    threshold = np.median(heights) * 0.3 if heights else 10
    sorted_boxes, cur_line, base_y = [], [boxes[0]], boxes[0][0][1]
    for box in boxes[1:]:
        if abs(box[0][1] - base_y) <= threshold:
            cur_line.append(box)
        else:
            sorted_boxes.extend(sorted(cur_line, key=lambda b: b[0][0]))
            cur_line, base_y = [box], box[0][1]
    if cur_line:
        sorted_boxes.extend(sorted(cur_line, key=lambda b: b[0][0]))
    return sorted_boxes

def run_ocr(img_cv2):
    result = Detector.ocr(img_cv2, cls=False)
    if result is None or len(result) == 0 or result[0] is None:
        return "", []
    boxes = result[0]
    if not boxes: return "", []
    boxes = Sort_Boxes(boxes)
    texts, box_data = [], []
    for x in boxes:
        crop = Crop_Padding(img_cv2, x)
        hc, wc = crop.shape[:2]
        if hc < 8 or wc < 8: continue
        text = recognizer.predict(Image.fromarray(crop)).strip()
        if len(text) > 1 and not text.isdigit():
            texts.append(text)
            box_data.append({"text": text, "area": wc * hc, "box": x})
    ocr_text = postprocess_ocr(" ".join(texts))
    return ocr_text, box_data
print('OCR pipeline ready')

## VLM Engine - vLLM API Client

In [ ]:
PROMPT_SUMMARY = """You are an expert analyzing Vietnamese FMCG product images.

## Task:
Write a short description in Vietnamese about the content of this image.

## MANDATORY Rules:
1. Describe the main context (product shot, banner, livestream, review photo, etc.).
2. Summarize the clearest text lines that appear in the image.
3. If brand or product names are present, embed them naturally into the description.
4. DO NOT hallucinate brands or products not visible. If unsure, ignore.
5. DO NOT try to read blurry or upside-down text. Skip those parts.
6. If the image has no text and no brands/products, return empty string.
7. Keep brand/product names in original characters (e.g., "Vinamilk", "TH True Milk").

## OCR text for reference (may contain minor errors):
__OCR_CONTEXT__

## Examples:
- "Anh chup hop sua Vinamilk Flex khong duong. Tren bao bi co dong chu Sua tuoi tiet trung."
- "Banner quang cao chuong trinh khuyen mai cua Nestle Milo voi dong chu Mua 2 tang 1."
- "Anh chup phong canh thien nhien, khong co san pham hoac nhan hang nao."
- (no text / no brand): ""

Write the description immediately:"""

def build_ocr_context(ocr_text, box_data=None):
    if not ocr_text or ocr_text.strip() in ["", " "]:
        return "No text detected."
    if not box_data:
        return 'Detected text: "' + ocr_text + '"'
    ranked = sorted(box_data, key=lambda b: b["area"], reverse=True)
    lines = []
    for i, b in enumerate(ranked[:10], 1):
        lines.append(str(i) + '. "' + b["text"] + '"')
    return "Detected text (by prominence):\n" + "\n".join(lines)

VLLM_BASE_URL = "http://localhost:25241/v1"
VLLM_MODEL = "Qwen/Qwen3-VL-4B-Instruct"
VLLM_CLIENT = OpenAI(base_url=VLLM_BASE_URL, api_key="EMPTY")
try:
    models = VLLM_CLIENT.models.list()
    print(f'vLLM connected: {[m.id for m in models]}')
except Exception:
    print(f"vLLM not reachable at {VLLM_BASE_URL}")

In [ ]:
def vlm_summarize(image_pil, ocr_text, box_data=None):
    try:
        context = build_ocr_context(ocr_text, box_data)
        prompt = PROMPT_SUMMARY.replace("__OCR_CONTEXT__", context)
        img = image_pil.copy()
        img.thumbnail((768, 768))
        buf = BytesIO()
        img.save(buf, format="JPEG", quality=85)
        b64 = base64.b64encode(buf.getvalue()).decode()
        resp = VLLM_CLIENT.chat.completions.create(
            model=VLLM_MODEL,
            messages=[{"role": "user", "content": [
                {"type": "image_url",
                 "image_url": {"url": "data:image/jpeg;base64," + b64}},
                {"type": "text", "text": prompt}]}],
            max_tokens=384, temperature=0.3, top_p=0.9)
        summary = resp.choices[0].message.content.strip()
        empty = {'""', "''", "none", "null", "n/a", "na", " ", ""}
        if summary.strip().lower() in empty: return ""
        return summary
    except Exception as e:
        print(f"[VLM Error] {e}")
        traceback.print_exc()
        return ""
print('vlm_summarize ready')

## Main Loop - OCR || VLM

In [ ]:
total = len(image_files)
print(f"Processing {total} images")

results, rlock = [], Lock()
CKPT = 100
err_cnt, elock = 0, Lock()
failed, flock = [], Lock()

def producer(tasks, q):
    global err_cnt
    for iid, ipath in tasks:
        try:
            img_pil = Image.open(ipath).convert("RGB")
            img_cv2 = preprocess(img_pil)
            text, boxes = run_ocr(img_cv2)
            q.put((iid, img_pil, text, boxes))
        except Exception as e:
            with elock: err_cnt += 1
            with flock: failed.append(iid)
            print(f"[P] {iid}: {e}")
            q.put((iid, None, "", None))
    q.put(None)

def consumer(q, bar):
    while True:
        item = q.get()
        if item is None: break
        iid, img_pil, text, boxes = item
        if img_pil is None:
            with rlock: results.append({"image_id": iid, "summary": ""})
            bar.update(1); continue
        try:
            summary = vlm_summarize(img_pil, text, boxes)
        except Exception as e:
            print(f"[C] {iid}: {e}"); summary = ""
        with rlock:
            results.append({"image_id": iid, "summary": summary})
            cc = len(results)
        bar.update(1)
        if cc % CKPT == 0:
            with rlock: snap = list(results)
            t = time.time() - start
            print(f"  CKPT {cc}/{total} | {t/max(1,cc):.2f}s/img | {err_cnt} err")
            with open("checkpoint.jsonl","w",encoding="utf-8") as f:
                for r in snap: f.write(json.dumps(r,ensure_ascii=False)+"\n")

tasks = [(os.path.splitext(f)[0], os.path.join(TEST_IMAGE_FOLDER, f)) for f in image_files]
print(f"Tasks: {len(tasks)}")

start = time.time()
q = Queue(maxsize=5)
bar = tqdm(total=len(tasks), desc="Processing")
p = Thread(target=producer, args=(tasks, q))
c = Thread(target=consumer, args=(q, bar))
p.start(); c.start()
p.join(); c.join()
bar.close()

elapsed = time.time() - start
print(f"\nDone: {len(results)} images in {elapsed:.1f}s")
print(f"Avg: {elapsed/max(1,len(results)):.2f}s/img")
print(f"TP:  {len(results)/max(1,elapsed):.2f} img/s")
if err_cnt > 0: print(f"Errors: {err_cnt} - {failed[:10]}...")

with open("submission.jsonl","w",encoding="utf-8") as f:
    for r in results: f.write(json.dumps(r,ensure_ascii=False)+"\n")
print(f"submission.jsonl written ({len(results)} lines)")